In [1]:
import sys
sys.path.insert(0, '..')


# Basic imports
import jax.numpy as np
import jax.random as jr
import jax.scipy as jsp
import jax
import numpy as onp
import pickle

jax.config.update("jax_enable_x64", True)


# Optimisation imports
import zodiax as zdx
import optax
import optimistix as optx
import equinox as eqx

import numpyro
import numpyro.distributions as dist

from numpyro.infer import (
    MCMC,
    NUTS
)

from numpyro.infer.initialization import (
    init_to_value
)


# dLux imports
import dLux as dl
import dLux.utils as dlu

#FITS 
from astropy.io import fits
from pathlib import Path
import pandas as pd

# Visualisation imports
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import PowerNorm, TwoSlopeNorm
import chainconsumer as cc


%matplotlib inline
plt.rcParams['image.cmap'] = 'inferno'
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = 'lower'
plt.rcParams['figure.dpi'] = 72
plt.rcParams["font.size"] = 24

from detectors import *
from apertures import *
from models import *
from stats import posterior
from fitting import *
from plotting import *
from spectra import *

import jax.tree_util as jtu
import interpax as ipx

def set_array(pytree):
    dtype = np.float64 if jax.config.x64_enabled else np.float32
    floats, other = eqx.partition(pytree, eqx.is_inexact_array_like)
    floats = jtu.tree_map(lambda x: np.array(x, dtype=dtype), floats)
    return eqx.combine(floats, other)

def to_numpy_tree(tree):
    return jtu.tree_map(
        lambda x: onp.asarray(x)
        if hasattr(x, "shape")
        else x,
        tree
    )

In [ ]:
def make_single_hmc_model( model_single, exposures_single, single_params ):

    # Keys used by the model
    position_keys = tuple( single_params.params[ "positions" ].keys() )

    spectrum_keys = tuple( single_params.params[ "spectrum" ].keys() )

    # BFGS solution
    pos0 = np.stack([ single_params.params[ "positions" ][key] for key in position_keys ])
    spectrum0 = np.stack([ single_params.params[ "spectrum" ][key] for key in spectrum_keys ])

    spectrum_scale = np.maximum( np.abs(spectrum0) * 0.10, 50.0 )

    # NumPyro model
    def single_hmc_model():

        positions = numpyro.sample( "positions", dist.Normal( pos0, 0.2 ).to_event(2) )
        spectrum = numpyro.sample( "spectrum", dist.Normal( spectrum0, spectrum_scale ).to_event(2) )
        sampled_params = ModelParams({ "positions": { key: positions[i] for i, key in enumerate( position_keys ) }, 
                                      "spectrum": { key: spectrum[i] for i, key in enumerate( spectrum_keys ) }, })

        # Telescope stays fixed
        mdl = sampled_params.inject( model_single )
        total_loss = 0.0

        for exp in exposures_single:
            loss_image = exp.fit.loglike( mdl, exp, return_im=True )
            total_loss += np.sum( loss_image )

        numpyro.factor( "log_likelihood", -total_loss )

    init_values = { "positions": pos0, "spectrum": spectrum0, }

    return ( single_hmc_model, init_values )

In [ ]:
def make_binary_hmc_model( model_binary, exposures_binary, binary_params ):

    # Get the dictionary keys used by the model
   
    position_keys = tuple( binary_params.params[ "positions" ].keys() )
    primary_keys = tuple( binary_params.params[ "primary_spectrum" ].keys() )
    secondary_keys = tuple( binary_params.params[ "secondary_spectrum" ].keys() )

    # BFGS solution = centre of HMC

    pos0 = np.stack([ binary_params.params[ "positions" ][key] for key in position_keys ])
    sep0 = float( onp.asarray( binary_params.params[ "separation" ] ) )
    pa0 = float( onp.asarray( binary_params.params[ "position_angle" ] ) )
    primary0 = np.stack([ binary_params.params[ "primary_spectrum" ][key] for key in primary_keys ])
    secondary0 = np.stack([ binary_params.params[ "secondary_spectrum" ][key] for key in secondary_keys ])

    # Spectrum proposal/prior widths
    primary_scale = np.maximum( np.abs(primary0) * 0.10, 50.0 )
    secondary_scale = np.maximum( np.abs(secondary0) * 0.10, 50.0 )

    # NumPyro model
  
    def binary_hmc_model():

        # Position in pixels
        positions = numpyro.sample( "positions", dist.Normal( pos0, 0.2 ).to_event(2) )

        # Positive separation
        separation = numpyro.sample( "separation", dist.LogNormal( np.log(sep0), 0.10 ) )

        # Position angle in degrees
        position_angle = numpyro.sample( "position_angle", dist.Normal( pa0, 5.0 ) )

        # FULL 5-coefficient spectrum
        primary_spectrum = numpyro.sample( "primary_spectrum", dist.Normal( primary0, primary_scale ).to_event(2) )
        secondary_spectrum = numpyro.sample( "secondary_spectrum", dist.Normal( secondary0, secondary_scale ).to_event(2) )

        # Turn sampled arrays back into model params

        sampled_params = ModelParams({
            "positions": { key: positions[i] for i, key in enumerate( position_keys ) },
            "separation": separation,
            "position_angle": position_angle,
            "primary_spectrum": { key: primary_spectrum[i] for i, key in enumerate( primary_keys ) },
            "secondary_spectrum": { key: secondary_spectrum[i] for i, key in enumerate( secondary_keys ) },
        })

        # Inject ONLY source parameters,Telescope remains fixed at BFGS.
        mdl = sampled_params.inject( model_binary )

        # Likelihood

        total_loss = 0.0

        for exp in exposures_binary:
            loss_image = exp.fit.loglike( mdl, exp, return_im=True )
            total_loss = ( total_loss + np.sum(loss_image) )

        # loss = negative log likelihood
        # NumPyro wants log likelihood
        numpyro.factor( "log_likelihood", -total_loss )

    # Start NUTS exactly at BFGS solution

    init_values = {
        "positions": pos0,
        "separation": sep0,
        "position_angle": pa0,
        "primary_spectrum": primary0,
        "secondary_spectrum": secondary0,
    }

    return ( binary_hmc_model, init_values )

In [ ]:

def summarise_chain(samples, method="median"):

    if method == "median":
        return np.median(samples, axis=0)

    elif method == "mean":
        return np.mean(samples, axis=0)

    else:
        raise ValueError(
            "method must be 'median' or 'mean'"
        )

In [ ]:
def make_hmc_comparison_plot( target, filt, single_mcmc, binary_mcmc, model_single, model_binary, exposures_single, exposures_binary, 
                             results_dir=None, summary_method="median", image_cmap="inferno", residual_cmap="RdBu_r", power_gamma=0.5, residual_percentile=99, 
                             save=True, show=False, ):
    
    """
   
    ROW 1
        Data | Single HMC model | Data - Single

    ROW 2
        Data | Binary HMC model | Data - Binary

    ROW 3
        Data | Binary primary only | Data - Primary

    The bottom row uses the PRIMARY COMPONENT from
    the binary posterior solution, rather than the
    independently fitted single-source solution.
    """
 
    # 1. EXPOSURES
    exp_single = exposures_single[0]
    exp_binary = exposures_binary[0]

   
    # 2. HMC SAMPLES
    single_samples = single_mcmc.get_samples()
    binary_samples = binary_mcmc.get_samples()

    sep = onp.asarray( binary_samples["separation"] )
    pa = onp.asarray( binary_samples["position_angle"] )

    print( "Separation:", onp.median(sep), "+/-", onp.std(sep), "pixels" )
    print( "Position angle:", onp.median(pa), "+/-", onp.std(pa), "degrees" )
    print( "Separation range:", sep.min(), "to", sep.max() )
    print( "PA range:", pa.min(), "to", pa.max() )

    fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axs[0].plot( onp.asarray(binary_samples["separation"]), marker="o" )
    axs[0].axhline( onp.mean(binary_samples["separation"]), linestyle="--" )
    axs[0].set_ylabel("Separation [pixels]")
    axs[0].set_title("HMC trace — separation")

    axs[1].plot( onp.asarray(binary_samples["position_angle"]), marker="o" )
    axs[1].axhline( onp.mean(binary_samples["position_angle"]), linestyle="--" )
    axs[1].set_ylabel("Position angle [deg]")
    axs[1].set_xlabel("HMC sample")
    axs[1].set_title("HMC trace — position angle")

    plt.tight_layout()
    plt.show()

   
    # 3. POSTERIOR SUMMARY
   
    # SINGLE
    single_pos = summarise_chain( single_samples["positions"], summary_method )[0]
    single_spec = summarise_chain( single_samples["spectrum"], summary_method )[0]

    # BINARY
    binary_pos = summarise_chain( binary_samples["positions"], summary_method )[0]
    binary_primary_spec = summarise_chain( binary_samples["primary_spectrum"], summary_method )[0]
    binary_secondary_spec = summarise_chain( binary_samples["secondary_spectrum"], summary_method )[0]
    binary_sep = summarise_chain( binary_samples["separation"], summary_method )
    binary_pa = summarise_chain( binary_samples["position_angle"], summary_method )


    # 4. SINGLE POSTERIOR PARAMETERS
   
    single_hmc_params = ModelParams({
        "positions": { exp_single.fit.get_key( exp_single, "positions" ): single_pos },
        "spectrum": { exp_single.fit.get_key( exp_single, "spectrum" ): single_spec }
    })


    # 5. BINARY POSTERIOR PARAMETERS
   
    position_key = exp_binary.fit.get_key( exp_binary, "positions" )
    primary_spec_key = exp_binary.fit.get_key( exp_binary, "primary_spectrum" )
    secondary_spec_key = exp_binary.fit.get_key( exp_binary, "secondary_spectrum" )


    binary_hmc_params = ModelParams({
        "positions": { position_key: binary_pos },
        "primary_spectrum": { primary_spec_key: binary_primary_spec },
        "secondary_spectrum": { secondary_spec_key: binary_secondary_spec },
        "separation": binary_sep,
        "position_angle": binary_pa,
    })


    # 6. GENERATE SINGLE + BINARY MODELS
  
    single_hmc_model = single_hmc_params.inject( model_single )
    binary_hmc_model = binary_hmc_params.inject( model_binary )

    single_image = onp.asarray( exp_single.fit( single_hmc_model, exp_single ) )
    binary_image = onp.asarray( exp_binary.fit( binary_hmc_model, exp_binary ) )


    
    # 7. PRIMARY-ONLY BINARY MODEL
   

    zero_secondary = np.zeros_like( binary_secondary_spec )

    primary_only_params = ModelParams({
        "secondary_spectrum": { secondary_spec_key: zero_secondary }
    })


    # Start from the COMPLETE binary posterior model
    # and change only the secondary spectrum.
    primary_only_model = primary_only_params.inject( binary_hmc_model )
    primary_image = onp.asarray( exp_binary.fit( primary_only_model, exp_binary ) )

    # 8. DATA + MASK
   
    data = onp.asarray( exp_single.data )

    err = onp.asarray( exp_single.err )

    bad = onp.asarray( exp_single.bad ).astype(bool)

    valid = (
        (~bad)
        & onp.isfinite(data)
        & onp.isfinite(err)
        & onp.isfinite(single_image)
        & onp.isfinite(binary_image)
        & onp.isfinite(primary_image)
        & (err > 0)
    )

    # 9. RESIDUALS

    single_residual = ( data - single_image )
    binary_residual = ( data - binary_image )
    primary_residual = ( data - primary_image )

    # 10. MASKED ARRAYS
   
    data_plot = onp.ma.array( data, mask=~valid )
    single_plot = onp.ma.array( single_image, mask=~valid )
    binary_plot = onp.ma.array( binary_image, mask=~valid )
    primary_plot = onp.ma.array( primary_image, mask=~valid )

    single_residual_plot = onp.ma.array( single_residual, mask=~valid )
    binary_residual_plot = onp.ma.array( binary_residual, mask=~valid )
    primary_residual_plot = onp.ma.array( primary_residual, mask=~valid )


    
    # 11. IMAGE COLOUR SCALE
    
    image_vmin = onp.nanpercentile( data[valid], 1 )
    image_vmax = onp.nanpercentile( data[valid], 99.8 )
    image_norm = PowerNorm( gamma=power_gamma, vmin=image_vmin, vmax=image_vmax )


    
    # 12. RESIDUAL COLOUR SCALE
    
    residual_values = onp.concatenate([ onp.abs( single_residual[valid] ).ravel(), onp.abs( binary_residual[valid] ).ravel(), onp.abs( primary_residual[valid] ).ravel(), ])
    resid_lim = onp.nanpercentile( residual_values, residual_percentile )
    residual_norm = TwoSlopeNorm( vmin=-resid_lim, vcenter=0, vmax=resid_lim )


    
    # 13. 3 x 3 FIGURE
   
    fig, axs = plt.subplots(
        3, 3, figsize=(15, 15),

        # Same pixel coordinates in every panel
        sharex=True,
        sharey=True,
        layout="compressed"
    )


    # ROW 1 — SINGLE
    
    im_data = axs[0, 0].imshow( data_plot, cmap=image_cmap, norm=image_norm )
    axs[0, 0].set_title( "NICMOS Data" )

    axs[0, 1].imshow( single_plot, cmap=image_cmap, norm=image_norm )
    axs[0, 1].set_title( "Single HMC Model" )

    im_residual = axs[0, 2].imshow( single_residual_plot, cmap=residual_cmap, norm=residual_norm )
    axs[0, 2].set_title( "Single Residual" )


    # ROW 2 — FULL BINARY
   
    axs[1, 0].imshow( data_plot, cmap=image_cmap, norm=image_norm )
    axs[1, 0].set_title( "NICMOS Data" )

    axs[1, 1].imshow( binary_plot, cmap=image_cmap, norm=image_norm )
    axs[1, 1].set_title( "Binary HMC Model" )

    axs[1, 2].imshow( binary_residual_plot, cmap=residual_cmap, norm=residual_norm )
    axs[1, 2].set_title( "Binary Residual" )


    # ROW 3 — PRIMARY COMPONENT FROM BINARY FIT
    
    axs[2, 0].imshow( data_plot, cmap=image_cmap, norm=image_norm )
    axs[2, 0].set_title( "NICMOS Data" )

    axs[2, 1].imshow( primary_plot, cmap=image_cmap, norm=image_norm )
    axs[2, 1].set_title( "Binary Primary Only" )

    axs[2, 2].imshow( primary_residual_plot, cmap=residual_cmap, norm=residual_norm )
    axs[2, 2].set_title( "Data − Binary Primary" )

   
    # 14. CLEAN AXES

    for ax in axs.flat:
        ax.tick_params( labelsize=11 )


    fig.supxlabel( "x pixel", fontsize=14 )
    fig.supylabel( "y pixel", fontsize=14 )


    # 15. COLORBARS
    
    image_cbar = fig.colorbar( im_data, ax=axs[:, :2], shrink=0.75, pad=0.02 )
    image_cbar.set_label( "Flux" )

    residual_cbar = fig.colorbar( im_residual, ax=axs[:, 2], shrink=0.75, pad=0.02 )
    residual_cbar.set_label( "Data − Model" )

    # 16. TITLE
    fig.suptitle( f"{target} - NICMOS {filt} — HMC Single vs Binary Comparison", fontsize=20 )

    
    # 17. SAVE
   
    if save and results_dir is not None:

        safe_target = ( str(target) .replace(" ", "_") .replace("/", "_") )
        plot_dir = ( Path(results_dir) / safe_target / filt / "plots" )
        plot_dir.mkdir( parents=True, exist_ok=True )

        save_path = ( plot_dir / "hmc_comparison.png" )
        fig.savefig( save_path, dpi=200, bbox_inches="tight" )
        print( "Saved plot:", save_path )


    if show:
        plt.show()

    else:
        plt.close(fig)


    # 18. RETURN EVERYTHING FOR AUTOMATION
   
    results = {

        "target": target,
        "filter": filt,
        "single_model_image": single_image,
        "binary_model_image": binary_image,
        "primary_only_model_image": primary_image,
        "single_residual": single_residual,
        "binary_residual": binary_residual,
        "primary_only_residual": primary_residual,
        "binary_separation": float( onp.asarray(binary_sep) ),
        "binary_position_angle": float( onp.asarray(binary_pa) ),
        "binary_primary_spectrum": onp.asarray( binary_primary_spec ),
        "binary_secondary_spectrum": onp.asarray( binary_secondary_spec ),
        "valid": valid,

    }


    return results

In [ ]:
def run_hmc_for_group( target, filt, exposures_single, exposures_binary, bfgs_output, optics, detector, config, results_dir, save=True, make_plot=True, show_plot=False ):

    print()
    print("=" * 60)
    print("RUNNING HMC")
    print("TARGET:", target)
    print("FILTER:", filt)
    print("=" * 60)

    # 1. Get BFGS solution
   
    single_bfgs_params = ModelParams( bfgs_output["single"]["params"] )
    binary_bfgs_params = ModelParams( bfgs_output["binary"]["params"] )
    print("Loaded BFGS parameters")

    # 2. Build models

    model_single = set_array( NICMOSModel( exposures_single, single_bfgs_params.params, optics, detector ) )
    model_binary = set_array( NICMOSModel( exposures_binary, binary_bfgs_params.params, optics, detector ) )

    # 3. HMC settings
    
    num_warmup = config["num_warmup"]
    num_samples = config["num_samples"]
    seed = config["seed"]

    print()
    print("HMC settings")
    print("Warmup :", num_warmup)
    print("Samples:", num_samples)
    print("Seed   :", seed)


    # Use separate random keys for single/binary
    key = jr.PRNGKey(seed)

    single_key, binary_key = jr.split( key, 2 )


    # 4. SINGLE HMC MODEL
   
    single_hmc_model, single_init_values = make_single_hmc_model( model_single, exposures_single, single_bfgs_params )

    # Start NUTS exactly at the BFGS solution
    single_kernel = NUTS( single_hmc_model, init_strategy=init_to_value( values=single_init_values ) )
    single_mcmc = MCMC( single_kernel, num_warmup=num_warmup, num_samples=num_samples, progress_bar=True )

    print()
    print("Running SINGLE HMC...")
    single_mcmc.run( single_key )

    # 5. BINARY HMC MODEL
   
    binary_hmc_model, binary_init_values = make_binary_hmc_model( model_binary, exposures_binary, binary_bfgs_params )
    binary_kernel = NUTS( binary_hmc_model, init_strategy=init_to_value( values=binary_init_values ) )
    binary_mcmc = MCMC( binary_kernel, num_warmup=num_warmup, num_samples=num_samples, progress_bar=True )

    print()
    print("Running BINARY HMC...")
    binary_mcmc.run( binary_key )

    # 6. Posterior samples
    
    single_samples = ( single_mcmc.get_samples() )
    binary_samples = ( binary_mcmc.get_samples() )

    print()
    print("Single samples:")

    for key, value in single_samples.items(): 
        print( key, value.shape )

    print()
    print("Binary samples:")

    for key, value in binary_samples.items():
        print( key, value.shape )


    # 7. Diagnostics
    
    single_extra = ( single_mcmc.get_extra_fields() )
    binary_extra = ( binary_mcmc.get_extra_fields() )


    single_divergences = 0
    if "diverging" in single_extra:
        single_divergences = int( onp.sum( onp.asarray( single_extra[ "diverging" ] ) ) )


    binary_divergences = 0
    if "diverging" in binary_extra:
        binary_divergences = int( onp.sum( onp.asarray( binary_extra[ "diverging" ] ) ) )


    print()
    print( "Single divergences:", single_divergences )
    print( "Binary divergences:", binary_divergences )

    print()
    print("SINGLE HMC SUMMARY")
    single_mcmc.print_summary()

    print()
    print("BINARY HMC SUMMARY")
    binary_mcmc.print_summary()

    # 8. STORE RESULTS
   
    hmc_output = {

        "target": target,
        "filter": filt,
        "num_warmup": num_warmup,
        "num_samples": num_samples,
        "seed": seed,
        "single": { "samples": to_numpy_tree( single_samples ), "divergences": single_divergences, },
        "binary": { "samples": to_numpy_tree( binary_samples ), "divergences": binary_divergences, },
    }

    # 9. MAKE HMC COMPARISON PLOT
  
    plot_result = None

    if make_plot:

        plot_result = make_hmc_comparison_plot(
            target=target,
            filt=filt,
            single_mcmc=single_mcmc,
            binary_mcmc=binary_mcmc,

            model_single=model_single,
            model_binary=model_binary,

            exposures_single=exposures_single,
            exposures_binary=exposures_binary,

            results_dir=results_dir,
            summary_method="median",

            save=True,
            show=False,
        )


        # Store useful plot-derived values
        hmc_output["plot"] = {
            "binary_separation": plot_result[ "binary_separation" ],
            "binary_position_angle": plot_result[ "binary_position_angle" ],
        }

   
    # 10. SAVE HMC OUTPUT
    
    if save:

        safe_target = ( str(target) .replace(" ", "_") .replace("/", "_") )
        group_dir = ( Path(results_dir) / safe_target / filt )
        group_dir.mkdir( parents=True, exist_ok=True )
        save_path = ( group_dir / "hmc.pkl" )
        with open( save_path, "wb" ) as f: pickle.dump( hmc_output, f )
        print( "Saved:", save_path )
        hmc_output["save_path"] = ( str(save_path) )
  

    # 11. RETURN
  
    return (
        hmc_output,
        single_mcmc,
        binary_mcmc,
        model_single,
        model_binary
    )